In [ ]:
import ee
import geemap
from pathlib import Path
import re
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.features import geometry_mask

In [ ]:
from tgbs_rs.config.config import (
    CURRENT_START, 
    CURRENT_END, 
    LAND_METRICS_DRIVE_FOLDER,
    LANDSCAPE_RASTER_DIR,
    DATA_DIR
)
from tgbs_rs.config.config import SITES_VIS_PARAMS, DW_BINARY_VIS_PARAMS
from tgbs_rs.data.sensors.dynamic_world.dynamic_world_preprocessing import build_annual_dw_woody_cover_collection
from tgbs_rs.utils import build_default_sites_featurecollection
from tgbs_rs.utils import buffer_sites_fc
from tgbs_rs.io import export_image_collection_to_drive

In [3]:
# Authenticate and Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project= "charrell-personal")

In [16]:
#Initialize Map
Map = geemap.Map()
Map = geemap.Map(height="800px")

In [4]:
# Build Site Feature Collection
sites_fc = build_default_sites_featurecollection()

# Buffer each feature by 0.5km
buffered_fc = buffer_sites_fc(sites_fc, buffer_m=500)
features = buffered_fc.getInfo()["features"]

##### Build Binary Woody vs. Non-Woody Landcover for each site

In [5]:
sites_dict = {}

for feat in features:
    props = feat["properties"]
    site_id = props["site_id"]

    buffered_feature = ee.Feature(feat)
    buffered_aoi = buffered_feature.geometry()

    annual_col = build_annual_dw_woody_cover_collection(
        aoi=buffered_aoi,
        start_date=CURRENT_START,
        end_date=CURRENT_END,
    )

    sites_dict[site_id] = {
        "feature": buffered_feature,
        "aoi": buffered_aoi,
        "site_id": props.get("site_id"),
        "site_name": props.get("site_name"),
        "site_category": props.get("site_category"),
        "collection": annual_col,
    }

##### Check Site IDs, size, and site collection years

In [ ]:
site_ids = list(sites_dict.keys())

for site in site_ids:
    col = sites_dict[site]["collection"]

    print(site, col.size().getInfo())
    print(col.aggregate_array("year").getInfo())

##### Select Each Site and Export Annual Collection

In [27]:
#site_info = sites_dict['ks_rehab']
#site_info = sites_dict['buda']
#site_info = sites_dict['gogoni']
#site_info = sites_dict['shimba_hills']
#site_info = sites_dict['degraded_1']
#site_info = sites_dict['degraded_2']
#site_info = sites_dict['degraded_3']

In [28]:
'''
tasks = export_image_collection_to_drive(
    collection=site_info["collection"],
    aoi=site_info["aoi"],
    folder=LANDSCAPE_METRICS_FOLDER,
    file_prefix=site_info['site_id'],
    scale=10,
    band_names=["woody_cover"],
    description_prefix=f"{site_info['site_id']}_dw_woody_cover",
    date_property=None,
    fallback_property="year",
    file_suffix="dw_woody_cover",
)
'''

In [ ]:
def build_buffered_sites_gdf(
    site_files: dict[str, str | Path],
    buffer_m: float = 500,
    target_crs: str = "EPSG:32637",
) -> gpd.GeoDataFrame:
    """
    Build a buffered GeoDataFrame from individual site files.

    The dictionary keys are used as the canonical `site_id` values so they
    match raster filename prefixes regardless of file attributes.
    """
    rows = []

    for site_id, path in site_files.items():
        gdf = gpd.read_file(path).to_crs(target_crs)
        geom = gdf.geometry.union_all().buffer(buffer_m)

        rows.append({"site_id": site_id, "geometry": geom})

    return gpd.GeoDataFrame(rows, crs=target_crs)


def calculate_valid_pixel_coverage_from_polygons(
    raster_dir: str | Path,
    sites_gdf: gpd.GeoDataFrame,
    pattern: str = "*.tif",
) -> pd.DataFrame:
    """
    Calculate valid-pixel coverage per site-year using buffered site polygons.

    The raster filename prefix before `_YYYY` is treated as `site_id`, and
    matched against `sites_gdf["site_id"]`, which should come from `site_files` keys.
    """
    raster_dir = Path(raster_dir)
    rows = []

    site_geom_map = dict(zip(sites_gdf["site_id"], sites_gdf.geometry))

    for path in sorted(raster_dir.glob(pattern)):
        match = re.match(r"(.+?)_(20\d{2})", path.stem)
        if not match:
            continue

        site_id = match.group(1)
        year = int(match.group(2))

        if site_id not in site_geom_map:
            print(f"Skipping {path.name}: no matching site_id '{site_id}' in site_files keys")
            continue

        with rasterio.open(path) as src:
            arr = src.read(1, masked=True)
            site_geom = gpd.GeoSeries([site_geom_map[site_id]], crs=sites_gdf.crs).to_crs(src.crs).iloc[0]

            aoi_mask = geometry_mask(
                [site_geom],
                transform=src.transform,
                invert=True,
                out_shape=(src.height, src.width),
            )

            total_pixels_in_aoi = int(np.count_nonzero(aoi_mask))
            valid_pixels_in_aoi = int(np.count_nonzero(aoi_mask & (~arr.mask)))
            valid_coverage = (
                valid_pixels_in_aoi / total_pixels_in_aoi
                if total_pixels_in_aoi > 0 else np.nan
            )

        rows.append(
            {
                "site_id": site_id,
                "year": year,
                "filename": path.name,
                "total_pixels_in_aoi": total_pixels_in_aoi,
                "valid_pixels_in_aoi": valid_pixels_in_aoi,
                "valid_coverage": valid_coverage,
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(["site_id", "year"])
        .reset_index(drop=True)
    )

In [52]:
site_files = {
    "ks_rehab": DATA_DIR / "ks_rehab_blocks.geojson",
    "buda": DATA_DIR / "buda_aoi.geojson",
    "gogoni": DATA_DIR / "gogoni_aoi.geojson",
    "shimba_hills": DATA_DIR / "shimba_hills_aoi.geojson",
    "degraded_1": DATA_DIR / "degraded_1_aoi.geojson",
    "degraded_2": DATA_DIR / "degraded_2_aoi.geojson",
    "degraded_3": DATA_DIR / "degraded_3_aoi.geojson",
}

In [53]:
sites_gdf = build_buffered_sites_gdf(site_files, buffer_m=500, target_crs="EPSG:32637")

In [54]:
coverage_df = calculate_valid_pixel_coverage_from_polygons(
    raster_dir=LANDSCAPE_RASTER_DIR,
    sites_gdf=sites_gdf,
)

In [55]:
coverage_df

,site_id,year,filename,total_pixels_in_aoi,valid_pixels_in_aoi,valid_coverage
0,buda,2018,buda_2018_dw_woody_cover.tif,129553,129549,0.999969
1,buda,2019,buda_2019_dw_woody_cover.tif,129553,118571,0.915232
2,buda,2020,buda_2020_dw_woody_cover.tif,129553,129549,0.999969
3,buda,2021,buda_2021_dw_woody_cover.tif,129553,129549,0.999969
4,buda,2022,buda_2022_dw_woody_cover.tif,129553,122984,0.949295
5,buda,2023,buda_2023_dw_woody_cover.tif,129553,129549,0.999969
6,buda,2024,buda_2024_dw_woody_cover.tif,129553,129549,0.999969
7,buda,2025,buda_2025_dw_woody_cover.tif,129553,129549,0.999969
8,degraded_1,2018,degraded_1_2018_dw_woody_cover.tif,49052,49050,0.999959
9,degraded_1,2019,degraded_1_2019_dw_woody_cover.tif,49052,39140,0.797929
